In [13]:
import sys
sys.path.insert(0, '/home/jazz/Projects/Statistical-Learning-e20452')

import pandas as pd
import numpy as np
from itables import init_notebook_mode
from open_dataset_store import quick_start
import plotly.express as px
store = quick_start('./ExperimentResults', backend='local')

init_notebook_mode(all_interactive=True)

# import itables.options as opt
# opt.lengthMenu = [10, 25, 50]
# opt.scrollX = True

CSV_PATH = './results/state_log.csv'

Store initialised at: ./ExperimentResults (Backend: local)


In [ ]:
df = pd.read_csv(CSV_PATH)
df = df.copy()
base_year = 2014
def _ep_to_datetime(row):
    day, hour, minute = int(row['DayOfYear']), int(row['Hour']), int(row['Minute'])
    if hour >= 24:
        day += 1
        hour -= 24
    return pd.Timestamp(year=base_year, month=1, day=1) + pd.Timedelta(days=day-1, hours=hour, minutes=minute)

df['Datetime'] = df.apply(_ep_to_datetime, axis=1)
df['timestamp'] = df['Datetime'].astype('int64') // 10**9
ZONES = ['SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1']
print(f'Loaded {len(df)} timesteps, columns: {len(df.columns)}')

# list(df.columns)
# df.head(n=20)
# sum = store.get_df_summary(df, detailed=True)


Loaded 768 timesteps, columns: 93


['DayOfYear',
 'Hour',
 'Minute',
 'SPACE1-1_EKF_Status',
 'SPACE2-1_EKF_Status',
 'SPACE3-1_EKF_Status',
 'SPACE4-1_EKF_Status',
 'SPACE5-1_EKF_Status',
 'AHU_Coordinator_Status',
 'Out_Temp_C',
 'Out_RH_pct',
 'Outdoor_Air_Temp_C',
 'Outdoor_Air_RH_pct',
 'Outdoor_Air_Flow_kg_s',
 'Outdoor_Air_CO2_ppm',
 'Relief_Air_Temp_C',
 'Relief_Air_RH_pct',
 'Relief_Air_Flow_kg_s',
 'Relief_Air_CO2_ppm',
 'Mixer_Inlet_Temp_C',
 'Mixer_Inlet_RH_pct',
 'Mixer_Inlet_Flow_kg_s',
 'Mixer_Inlet_CO2_ppm',
 'Mixed_Air_Temp_C',
 'Mixed_Air_RH_pct',
 'Mixed_Air_Flow_kg_s',
 'Mixed_Air_CO2_ppm',
 'CC_Out_Temp_C',
 'CC_Out_RH_pct',
 'CC_Out_Flow_kg_s',
 'CC_Out_CO2_ppm',
 'HC_Out_Temp_C',
 'HC_Out_RH_pct',
 'HC_Out_Flow_kg_s',
 'HC_Out_CO2_ppm',
 'Fan_Out_Temp_C',
 'Fan_Out_RH_pct',
 'Fan_Out_Flow_kg_s',
 'Fan_Out_CO2_ppm',
 'SPACE1-1_Temp_C',
 'SPACE1-1_T_m_C',
 'SPACE1-1_W_in_kg_kg',
 'SPACE1-1_RH_pct',
 'SPACE1-1_VAV_Flow_kg_s',
 'SPACE1-1_Reheater_W',
 'SPACE1-1_CO2_ppm',
 'SPACE1-1_Occupants',
 'SPACE

In [16]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------------------------------------------------
# 1. AHU Monitoring (Temperature, Humidity, Flow, CO2)
# ---------------------------------------------------------
def plot_ahu_monitoring(df):
    """
    Creates subplots to monitor Temperature, Humidity, Flow, and CO2 
    at various points across the Air Handling Unit.
    """
    # Define the stages we want to track
    stages = ['Outdoor_Air', 'Relief_Air', 'Mixer_Inlet', 'Mixed_Air', 'CC_Out', 'HC_Out', 'Fan_Out']
    
    fig = make_subplots(
        rows=4, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.05,
        subplot_titles=('Temperatures (°C)', 'Relative Humidity (%)', 'Air Flow (kg/s)', 'CO2 Levels (ppm)')
    )

    # 1. Temperature
    for stage in stages:
        col = f'{stage}_Temp_C'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=1, col=1)

    # 2. Humidity
    for stage in stages:
        col = f'{stage}_RH_pct'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=2, col=1)

    # 3. Flow
    for stage in stages:
        col = f'{stage}_Flow_kg_s'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=3, col=1)

    # 4. CO2
    for stage in stages:
        col = f'{stage}_CO2_ppm'
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=4, col=1)

    fig.update_layout(height=1000, title_text="AHU System Monitoring across Stages", hovermode="x unified")
    fig.show()


# ---------------------------------------------------------
# 2. Zone Monitoring (Select a specific zone)
# ---------------------------------------------------------
def plot_zone_monitoring(df, zone_name="SPACE1-1"):
    """
    Plots all relevant metrics for a specifically chosen zone.
    Example zone_names: 'SPACE1-1', 'SPACE2-1', 'SPACE3-1', 'SPACE4-1', 'SPACE5-1'
    """
    # Extract columns that belong to the selected zone
    zone_cols = [col for col in df.columns if zone_name in col]
    
    # Group them by metric type for better visualization
    temp_cols = [c for c in zone_cols if 'Temp_C' in c or 'T_m_C' in c]
    rh_cols = [c for c in zone_cols if 'RH_pct' in c]
    flow_cols = [c for c in zone_cols if 'Flow' in c]
    co2_cols = [c for c in zone_cols if 'CO2' in c]
    power_cols = [c for c in zone_cols if 'Load_W' in c or 'Reheater_W' in c]
    occupant_cols = [c for c in zone_cols if 'Occupants' in c]
    
    fig = make_subplots(
        rows=5, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.04,
        subplot_titles=(
            f'{zone_name} Temperatures', 
            f'{zone_name} Humidity & CO2', 
            f'{zone_name} Air Flow', 
            f'{zone_name} Power (Reheat & Equip)',
            f'{zone_name} Occupancy'
        )
    )

    # Temp
    for col in temp_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=1, col=1)
        
    # RH & CO2 (plotting together on dual y-axes is complex in subplots, so plotting separate traces)
    for col in rh_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=2, col=1)
    for col in co2_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, line=dict(dash='dot')), row=2, col=1)
        
    # Flow
    for col in flow_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=3, col=1)
        
    # Power
    for col in power_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col), row=4, col=1)
        
    # Occupants
    for col in occupant_cols:
        fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, fill='tozeroy'), row=5, col=1)

    fig.update_layout(height=1200, title_text=f"Comprehensive Log for {zone_name}", hovermode="x unified")
    fig.show()


# ---------------------------------------------------------
# 3. Energy Consumption (Instantaneous and Cumulative)
# ---------------------------------------------------------
def plot_energy_consumption(df):
    """
    Plots the interval (instantaneous) energy meters and calculates/plots 
    the cumulative energy consumption over time.
    """
    energy_cols = ['Meter_Bldg_Elec_J', 'Meter_HVAC_Elec_J', 'Meter_AHU_Elec_J', 'Meter_Bldg_Gas_J']
    
    fig = make_subplots(
        rows=2, cols=1, 
        shared_xaxes=True, 
        vertical_spacing=0.1,
        subplot_titles=('Interval Energy Consumption (Joules)', 'Cumulative Energy Consumption (Joules)')
    )

    # 1. Instantaneous / Interval
    for col in energy_cols:
        if col in df.columns:
            fig.add_trace(go.Scatter(x=df['timestamp'], y=df[col], name=col, mode='lines'), row=1, col=1)

    # 2. Cumulative (using pandas .cumsum() to calculate running total)
    for col in energy_cols:
        if col in df.columns:
            cumulative_series = df[col].cumsum()
            fig.add_trace(go.Scatter(x=df['timestamp'], y=cumulative_series, name=f'{col} (Cumulative)', mode='lines'), row=2, col=1)

    fig.update_layout(height=700, title_text="Building Energy Meters", hovermode="x unified")
    fig.show()

In [17]:
plot_ahu_monitoring(df)

In [18]:
plot_zone_monitoring(df, zone_name="SPACE1-1")

In [19]:
plot_energy_consumption(df)